# 05 · The experiment — turning answers into a defensible claim

**The thesis: a single run is an anecdote. A claim needs repeats, a spread, and
a cost.** This notebook shows the harness that produces the paper's numbers and
the three rules that keep them honest.

Needs a model key. The run here is deliberately tiny so it is cheap to execute;
the real run is the same command with bigger numbers.

> **Note on saved outputs.** The cells that call a model are intentionally left
> unexecuted in this commit: at the time of writing both configured providers were
> unavailable (OpenAI on a free-tier daily cap, Groq suspended for billing). Every
> cell that does *not* need a model has been run and its real output is saved.
> Re-run this notebook once a key is live and the remaining outputs will fill in.

## What this notebook demonstrates

| Name | What it does | Example |
|---|---|---|
| `run_eval.py` | run configurations over the benchmark and score them | `--configs baseline,agent` |
| `--repeats N` | run everything N times to measure the spread | `--repeats 3` |
| `--sample-seed` | random representative sample, not the first N | `--sample-seed 7` |
| `BUDGET_USD` | hard spend ceiling; stops mid-run and saves partials | `BUDGET_USD=2.0` |

## Step 1 — the three rules

```
 1. SAMPLE PROPERLY   file order groups questions by source table, so the first
                      N can all come from one table. Use --sample-seed.
 2. REPEAT            the same question asked twice can get different answers.
                      Report mean +/- spread, never a single draw.
 3. NEVER CLAIM BELOW THE NOISE
                      if two configs differ by less than the run-to-run spread,
                      there is no difference. Say so.
```

A fourth, practical one: every run is capped by `BUDGET_USD`, so a mistake
costs cents rather than dollars.

## Step 2 — confirm provider and the spend ceiling before running

In [ ]:
import os, sys; sys.path.insert(0, "")
import agent as base
base._load_env_file()
p = base.resolve_provider()
print("provider :", p, "| model:", base.OPENAI_MODEL if p=="openai" else base.GROQ_MODEL)
print("ceiling  : $", os.environ.get("BUDGET_USD", "2.0 (default)"))

## Step 3 — a tiny ablation: baseline vs agent

Six questions, sampled with a seed so the mix is representative. This is
deliberately too small to conclude anything — it is here so you can see the
shape of the output.

In [ ]:
import subprocess, sys
cmd = [sys.executable, "run_eval.py",
       "--configs", "baseline,agent", "--limit", "6",
       "--sample-seed", "7", "--grounded-only"]
r = subprocess.run(cmd, capture_output=True, text=True, env={**os.environ, "BUDGET_USD": "2.0"})
print(r.stdout[-2600:])
if r.returncode: print("STDERR:", r.stderr[-700:])

## Step 4 — how to read that output

```
 accuracy block   overall, then split Lookup vs Compute. Always both.
 cost block       calls per question, $ per question, accuracy per dollar.
                  This is what the paper's title asks about: does the agentic
                  overhead pay for itself?
 DELTA line       the difference between configs. Compare it against the
                  spread before believing it.
```

## Step 5 — the real run

What Laasya runs for the paper. Three repeats is the minimum that gives a
spread; two domains supports the generality claim.

```bash
# main matrix (Biology)
BUDGET_USD=5 python run_eval.py --configs baseline,cot,agent \
    --limit 460 --repeats 3 --sample-seed 7

# transfer arm (a second domain)
BUDGET_USD=5 python run_eval.py --domain CompSci --configs baseline,agent \
    --limit 400 --repeats 3 --sample-seed 7
```

Run it as separate single-repeat jobs rather than one long job, so a crash does
not lose everything. Commit `results/*.csv` before re-running — the harness
overwrites them.

## Step 6 — what makes a result publishable

```
 · every number traces to a committed CSV
 · the model, the scorer, the temperature and N are stated next to it
 · the spread is reported, and nothing smaller than it is claimed
 · the spend is reported
 · a NEGATIVE result is reported exactly as found
```

The last one matters. If tools do not help, that is a finding — it tells other
engineers not to spend effort there. An overstated positive gets a paper
rejected; an honest negative does not.

## What to carry out of this notebook

```
 · sample with a seed, repeat at least 3x, cap the spend
 · report Lookup and Compute separately, always
 · a difference smaller than the spread is not a difference
 · cost is a result, not an afterthought
```